<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_02_die_hard_ic.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 02 — The Die, with the Initial Condition **Built In**

**Paired with L7.2 · Fundamental PDEs**

Notebook 01 found the error concentrated at $t = 0$, where the answer was
known exactly. This notebook stops giving that away.

Write the solution so it reproduces the initial field by construction:

$$\hat\theta(x,y,t) \;=\; \theta_0(x,y) \;+\; t\,D(x,y)\,N(x,y,t)$$

At $t = 0$ the second term vanishes and $\hat\theta = \theta_0$ **exactly**.
$D$ vanishes on the four edges, so the boundary condition is exact too — for
all $t$, including $t=0$, where $\theta_0$ already satisfies it.

Both soft terms disappear. One loss, no weights.

## What you will do

1. Build the trial solution and check both conditions before training.
2. Train on the residual alone.
3. Compare against notebook 01 instant by instant.
4. Find what the construction costs — because it does cost something.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The trial solution

### Your turn

In [ ]:
# TODO: build it, then check both conditions on an UNTRAINED network.
#
#   def theta_trial(model, xyt):
#       x, y, t = xyt[:, 0:1], xyt[:, 1:2], xyt[:, 2:3]
#       theta0 = pb.heat_initial(x, y)
#       D = pb.hard_bc_factor(x, y, pb.HEAT_DOMAIN)
#       return theta0 + t * D * model(xyt)
#
#   set_seed(0); probe = MLP(n_in=3, n_hidden=40, n_layers=4)
#
#   ic_err   : on initial_points(400, pb.HEAT_DOMAIN, t0=0.0, seed=3),
#              max |theta_trial - pb.heat_initial|
#   edge_err : on boundary_points_in_time(25, 10, pb.HEAT_DOMAIN,
#                                         (0.0, pb.HEAT_T_END), seed=3),
#              max |theta_trial|
#
# pb.heat_initial works on tensors -- it dispatches on type internally.

raise NotImplementedError("Build theta_trial and check both conditions")

In [ ]:
print(f"initial condition, untrained : {ic_err:.3e} K")
print(f"edge condition,    untrained : {edge_err:.3e} K")
check("initial field is exact before training", ic_err, 0.0, tol=1e-12)
check("edges are exact before training", edge_err, 0.0, tol=1e-12)

**What you should see.** Two `PASS` lines. A network that knows nothing already
reproduces the initial field and the edges to machine precision.

Compare with notebook 01, which needed four thousand Adam steps to get within
a fraction of a kelvin at $t = 0$ and never reached zero.

---

## 2 · One loss

### Your turn

In [ ]:
# TODO: residual and loss, with no boundary or initial terms.
#
#   def heat_residual_hard(model, xyt):
#       theta = theta_trial(model, xyt)
#       theta_t = grad(theta, xyt)[:, 2:3]
#       lap = d2(theta, xyt, 0) + d2(theta, xyt, 1)
#       return theta_t - pb.ALPHA * lap
#
#   k_slow, k_fast = pb.heat_rates()
#   R_SCALE = (pb.A_SLOW + pb.A_FAST) * k_fast
#
#   def make_loss_hard(model, xyt_f):
#       def loss():
#           return mse(heat_residual_hard(model, xyt_f) / R_SCALE)
#       return loss

raise NotImplementedError("Write heat_residual_hard and make_loss_hard")

## 3 · Train

In [ ]:
N_F = 4000

set_seed(88)
model_hard = MLP(n_in=3, n_hidden=40, n_layers=4)
describe(model_hard, N_F)

xyt_f = to_tensor(spacetime_points(N_F, pb.HEAT_DOMAIN, (0.0, pb.HEAT_T_END), seed=1),
                  requires_grad=True)

history_hard = train_two_stage(model_hard, make_loss_hard(model_hard, xyt_f),
                               adam_steps=4000, lbfgs_steps=200, lr=1e-3)
plot_curves(history_hard, title="the die — hard IC and BC, one loss term")
plt.show()

## 4 · Instant by instant, against notebook 01

### Your turn

In [ ]:
# TODO: score the hard model at the same instants and load notebook 01.
#
#   TIMES = [0.0, 0.005, 0.02, 0.05, 0.10, 0.20]
#   Score exactly as in notebook 01, but through theta_trial rather than the
#   bare model. Put the results in when_hard = {"rel": [...], "max": [...]}.
#
#   soft = np.load(os.path.join("Ex07.2_outputs", "nb01_die_soft.npz"))

raise NotImplementedError("Score the hard model and load notebook 01")

In [ ]:
print(error_table(
    [[f"{t*1e3:.0f}", f"{s:.3e}", f"{h:.3e}",
      "-" if h == 0 else f"{s/h:.1f}x"]
     for t, s, h in zip(TIMES, soft["rel"], when_hard["rel"])],
    ["t [ms]", "soft", "hard", "soft/hard"]))

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.semilogy(np.array(TIMES)*1e3, soft["rel"], "o-", lw=1.9, ms=6,
            color="#d94f2b", label="soft IC and BC")
ax.semilogy(np.array(TIMES)*1e3, when_hard["rel"], "s-", lw=1.9, ms=6,
            color="#1f77b4", label="hard IC and BC")
ax.set_xlabel("t [ms]"); ax.set_ylabel("relative L2")
ax.set_title("Where the two methods differ")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** The hard model's error at $t = 0$ is essentially zero
— it is the initial field, exactly — and the gap between the two narrows as
$t$ grows.

That shape is the honest summary. **Hard enforcement helps most where the
information was, and helps least far from it.** By 200 ms the network has been
integrating its own residual for three time constants and the free gift at
$t = 0$ is a long way behind it.

---

## 5 · What it cost

Two things, and the notebook should say both out loud.

**You had to know $\theta_0$ as a formula.** Here it is a sum of two modes. If
the initial field came from a measurement — a thermal camera frame, say — you
would need to interpolate it into something differentiable, and the derivatives
of that interpolant now sit inside your residual.

**The factor $t\,D$ shapes the solution everywhere, not just at the
boundaries.** It grows linearly in time, so the network must undo that growth
to represent a decaying field, and its own output has to fall roughly like
$e^{-kt}/t$ at late times.

### Your turn

In [ ]:
# TODO: look at what the network itself had to learn.
#
#   Evaluate the raw network N (not theta_trial) on the midplane
#   x = y = L/2 across the time window, and plot it against the trial solution.
#
#   ts = np.linspace(1e-4, pb.HEAT_T_END, 200)
#   q  = np.stack([np.full_like(ts, pb.L_DIE/2),
#                  np.full_like(ts, pb.L_DIE/2), ts], axis=1)
#   with torch.no_grad():
#       raw   = to_numpy(model_hard(to_tensor(q))).ravel()
#       full  = to_numpy(theta_trial(model_hard, to_tensor(q))).ravel()
#   exact = pb.heat_exact(q[:, 0], q[:, 1], ts)
#
#   Record them as raw, full, exact.

raise NotImplementedError("Inspect the raw network output")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.2))
axes[0].plot(ts*1e3, full, lw=2.0, color="#1f77b4", label="θ̂ = θ₀ + t D N")
axes[0].plot(ts*1e3, exact, lw=1.4, ls="--", color="#111111", label="exact")
axes[0].set_xlabel("t [ms]"); axes[0].set_ylabel("θ at the centre [K]")
axes[0].set_title("The trial solution does the right thing")
axes[0].legend(frameon=False, fontsize=9); axes[0].grid(alpha=0.25)

axes[1].plot(ts*1e3, raw, lw=2.0, color="#7b61a8")
axes[1].set_xlabel("t [ms]"); axes[1].set_ylabel("raw network output N")
axes[1].set_title("...and this is what N had to learn to make it happen")
axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

**What you should see.** A clean decaying curve on the left, and on the right
something that blows up as $t \to 0$ and flattens later.

That is the cost made visible. The trial solution is well behaved; the network
inside it is not. Dividing a bounded field by $t$ near the origin asks the
network to represent something with a steep feature it never had to have.

This is the general shape of the trade in every hard-enforcement construction:
**the condition becomes exact and the function the network must learn becomes
harder.** Whether that is worth it is a question about your problem, not a rule.

---

## 6 · Save

In [ ]:
path = os.path.join("Ex07.2_outputs", "nb02_die_hard.npz")
np.savez(path, times=np.asarray(TIMES),
         rel=np.asarray(when_hard["rel"], dtype=float),
         max_err=np.asarray(when_hard["max"], dtype=float),
         ic_err=ic_err, edge_err=edge_err,
         adam=history_hard["adam"], lbfgs=history_hard["lbfgs"])
torch.save(model_hard.state_dict(), os.path.join("Ex07.2_outputs", "nb02_die_hard.pt"))
print("wrote", path)

## 7 · Before you move on

1. The untrained network already satisfied both conditions. Say precisely why,
   term by term in the trial solution.
2. The advantage over soft enforcement shrank with time. Explain the mechanism.
3. Look again at the raw network output. What would you expect to go wrong if
   the time window were ten times longer?
4. The initial field here is a formula. Describe what you would do if it were
   a measured thermal image, and what new error you would be introducing.

Next: **notebook 03**, the panel — where the number of initial conditions stops
being a formality.